In [ ]:
include( "../geom.jl" )

include( "../0-params/rho-list.jl" )
include( "../0-params/nos-list.jl" )

using Optim
using ThreadsX

home, base = expfolder( "system" )

colormap = cgrad( :ice; rev=true )

In [ ]:
# Hyper parameters.
N = 1024;  s = 0.251

# Create folder and extract critical noise.
folderlist = [findfolder( N, ρ, s; base=base, crit=true ) for ρ ∈ ρlist]

# Extract critical noise values.
ηclist = first.( readdlm.( folderlist.*"../order-0.5.txt" ) );

In [ ]:
# Initialize progress meter.
meter = Progress( Nρ, "Importing data."; dt=1.0 )

# Initialize data variables.
Alist = Vector{Dict{defInt,Vector{defFloat}}}( undef, Nρ )
ℓlist = Vector{Dict{defInt,Vector{defFloat}}}( undef, Nρ )

# Import cluster dynamical data.
@threads for i ∈ 1:Nρ
    Alist[i] = loaddict( folderlist[i]*"convex-hull-area.txt", defFloat )
    ℓlist[i] = loaddict( folderlist[i]*"convex-hull-perimeter.txt", defFloat )
    next!( meter )
end

In [ ]:
# Compute mean and standard deviation.
ιmin = 2
ιdata = [[ι for ι in ιmin:length( A ) if !isempty( A[ι] )] for A ∈ Alist]
ιmax = round( log10( N ) )

# Area.
μAdata = [[mean( Alist[i][ι] ) for ι ∈ ιdata[i]] for i ∈ 1:Nρ]
σAdata = [[std(  Alist[i][ι] ) for ι ∈ ιdata[i]] for i ∈ 1:Nρ]

# Perimeter.
μℓdata = [[mean( ℓlist[i][ι] ) for ι ∈ ιdata[i]] for i ∈ 1:Nρ]
σℓdata = [[std(  ℓlist[i][ι] ) for ι ∈ ιdata[i]] for i ∈ 1:Nρ];

In [ ]:
# Extract the x and y-axis variables for fitting (area).
nmax = round( defInt, 1e6 )
xAlist = [vcat( [fill( ι, length( Alist[i][ι] ) ) for ι in ιdata[i]]...) for i ∈ 1:Nρ]
yAlist = [vcat( [Alist[i][ι] for ι in ιdata[i]]...) for i ∈ 1:Nρ]

# Perimeter.
xℓlist = [vcat( [fill( ι, length( ℓlist[i][ι] ) ) for ι in ιdata[i]]...) for i ∈ 1:Nρ]
yℓlist = [vcat( [ℓlist[i][ι] for ι in ιdata[i]]...) for i ∈ 1:Nρ];

In [ ]:
# Remove any zero values.
κdata = [(0.0 .< yAlist[i]) .& (0.0 .< yℓlist[i]) for i ∈ 1:Nρ]
xAlist = [xAlist[i][κdata[i]] for i ∈ 1:Nρ]
yAlist = [yAlist[i][κdata[i]] for i ∈ 1:Nρ]
xℓlist = [xℓlist[i][κdata[i]] for i ∈ 1:Nρ]
yℓlist = [yℓlist[i][κdata[i]] for i ∈ 1:Nρ];

In [ ]:
# Perform power law fits.
βℓlist = [fitpolylog( xℓlist[i], yℓlist[i]; cut=2 ) for i ∈ 1:Nρ]
βAlist = [(yℓlist[i].^2)\yAlist[i] for i ∈ 1:Nρ]

println( "power law for area: ", βℓlist )
println( "coefficient for perimeter: ", (1/(4π), βAlist) )

In [ ]:
i = 21

# Plot the trend against the mean of the sizes.
plt = plot(; size=(400,200), legend=false )

scatter!( plt, ιdata[i], μℓdata[i]; color=:gray )
plot!( plt, 1:N, polyloglaw.( 1:N; β=βℓlist[i] ); color=:cornflowerblue, lw=2 )

plot!( plt; xlims=(1,N), xscale=:log10 )
plot!( plt; ylims=10.0.^(-1,2), yscale=:log10 )

In [ ]:
i = 21

# Plot the trend against the mean of the sizes.
plt = plot(; size=(400,200), legend=false )

scatter!( plt, ιdata[i], μAdata[i]; color=:gray )
plot!( plt, 1:N, (1/4π).*polyloglaw.( 1:N; β=βℓlist[i] ).^2; color=:cornflowerblue, lw=2 )

plot!( plt; xlims=(1,N), xscale=:log10 )
plot!( plt; ylims=10.0.^(-2,3), yscale=:log10 )

In [ ]:
# Save parameter data.
if false
    # Save the cluster change parameters for each parameter case.
    for i ∈ 1:Nρ
        writedlm( folderlist[i]*"mean-convex-hull-area.txt", [μAdata[i] σAdata[i]] )
        writedlm( folderlist[i]*"mean-convex-hull-perimeter.txt", [μℓdata[i] σℓdata[i]] )
        writedlm( folderlist[i]*"area-perimeter-scaling.txt", [βℓlist[i]; βAlist[i]] )
    end
end